# 🫁 Pneumonia X-Ray Classifier — ConvNeXt V2 Transfer Learning

| Feature | Detail |
|---|---|
| Model | ConvNeXt V2 Tiny (ImageNet-22k pretrained) |
| Input | RGB, ImageNet normalization, 224×224 |
| Strategy | Frozen head → full unfreeze at epoch 5 |
| Balancing | Class-weighted CrossEntropyLoss (auto-computed) |
| Split | Augmentation-aware group split — zero leakage |
| Checkpoints | best.pt + latest.pt saved to Google Drive every epoch |
| Outputs | Training curves, confusion matrix, ROC curve, Grad-CAM |

> **Before running:** `Runtime → Change runtime type → T4 GPU`


## 📦 Step 1 — Install Dependencies

In [20]:
!pip install -q timm grad-cam scikit-learn seaborn tqdm
import sys, torch
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else '❌ No GPU — change runtime!'}")


Python  : 3.12.13
PyTorch : 2.10.0+cu128
GPU     : Tesla T4


## 📁 Step 2 — Mount Google Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive mounted")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Google Drive mounted


## ⚙️ Step 3 — Configuration

**Only edit this cell.**

Dataset structure expected:
```
your_dataset/
    NORMAL/      ← chest X-ray images
    PNEUMONIA/   ← chest X-ray images
```


In [22]:
from pathlib import Path
import os

# ─── PATHS ────────────────────────────────────────────────────────────────────
DATA_DIR   = "/content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset"   # ← YOUR dataset folder
OUTPUT_DIR = "/content/drive/MyDrive/ExecutionOutputs/pneumonia_outputs"   # ← where results are saved
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─── MODEL ────────────────────────────────────────────────────────────────────
MODEL_NAME = "convnextv2_base.fcmae_ft_in1k"

# ─── HYPERPARAMETERS ──────────────────────────────────────────────────────────
EPOCHS      = 30      # total training epochs
BATCH_SIZE  = 32      # reduce to 16 if GPU OOM
LR          = 3e-4    # learning rate for head-only phase
IMG_SIZE    = 224     # input resolution
UNFREEZE_EP = 5       # epoch to unfreeze full backbone (LR drops 10x)

# ─── DATA SPLITS ──────────────────────────────────────────────────────────────
# No manual split needed — dataset already has train/val/test folders
NUM_WORKERS = 2

# ─── MISC ─────────────────────────────────────────────────────────────────────
GRADCAM_SAMPLES = 8   # must be divisible by number of classes (2)
SEED            = 42

# ─── Verify Drive is mounted and create output dirs ───────────────────────────
if OUTPUT_DIR.startswith("/content/drive") and not os.path.exists("/content/drive/MyDrive"):
    raise RuntimeError("❌ Google Drive not mounted — run Step 2 first")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR, "checkpoints").mkdir(parents=True, exist_ok=True)

print("✅ Config ready")
print(f"   Data      : {DATA_DIR}")
print(f"   Outputs   : {OUTPUT_DIR}")
print(f"   Model     : {MODEL_NAME}")
print(f"   Epochs    : {EPOCHS}  |  Unfreeze at: {UNFREEZE_EP}  |  LR: {LR}")


✅ Config ready
   Data      : /content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset
   Outputs   : /content/drive/MyDrive/ExecutionOutputs/pneumonia_outputs
   Model     : convnextv2_base.fcmae_ft_in1k
   Epochs    : 30  |  Unfreeze at: 5  |  LR: 0.0003


## 📚 Step 4 — Imports & Reproducibility

In [23]:
import os, re, random, time, warnings
from pathlib import Path
from collections import defaultdict

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
import timm
from tqdm.notebook import tqdm

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_AVAILABLE = True
    print("✅ Grad-CAM available")
except ImportError:
    GRADCAM_AVAILABLE = False
    print("⚠️  Grad-CAM not found — re-run Step 1")

warnings.filterwarnings("ignore")

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device : {DEVICE}" + (f"  ({torch.cuda.get_device_name(0)})" if DEVICE.type=='cuda' else ""))


✅ Grad-CAM available
✅ Device : cuda  (Tesla T4)


## 🗂️ Step 5 — Data Loading & Augmentation-Aware Split

Splits at the **root image group level** — all `_aug_N` variants of the same original always stay in the same split. A leakage report is printed every run.


In [24]:
# ── Diagnose dataset folder structure ────────────────────────────────────────
import os
from pathlib import Path
from collections import defaultdict

print(f"Scanning: {DATA_DIR}\n")

for split in ["train", "val", "test"]:
    split_dir = Path(DATA_DIR) / split
    if not split_dir.exists():
        print(f"❌ Missing folder: {split_dir}")
        continue
    print(f"📁 {split}/")
    for cls_dir in sorted(split_dir.iterdir()):
        if not cls_dir.is_dir():
            continue
        files     = list(cls_dir.iterdir())
        ext_counts = defaultdict(int)
        for f in files:
            ext_counts[f.suffix.lower()] += 1
        ext_str = "  ".join(f"{ext}: {count}" for ext, count in sorted(ext_counts.items()))
        print(f"   {cls_dir.name}/  ({len(files)} files)   [{ext_str if ext_str else 'EMPTY'}]")
    print()


Scanning: /content/drive/MyDrive/ExecutionOutputs/pneumonia_dataset

📁 train/
   Normal/  (2991 files)   [.png: 2991]
   Pneumonia/  (2991 files)   [.png: 2991]

📁 val/
   Normal/  (237 files)   [.png: 237]
   Pneumonia/  (641 files)   [.png: 641]

📁 test/
   Normal/  (238 files)   [.png: 238]
   Pneumonia/  (641 files)   [.png: 641]



In [25]:
def get_transforms(img_size, is_train=True):
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    if is_train:
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomCrop(img_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean, std),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])


def build_dataloaders(data_dir, img_size, batch_size, num_workers):
    """
    Reads pre-split dataset directly from existing folder structure:
        data_dir/train/NORMAL
        data_dir/train/PNEUMONIA
        data_dir/val/NORMAL
        data_dir/val/PNEUMONIA
        data_dir/test/NORMAL
        data_dir/test/PNEUMONIA
    No manual splitting needed — uses the provided splits as-is.
    """
    train_dir = Path(data_dir) / "train"
    val_dir   = Path(data_dir) / "val"
    test_dir  = Path(data_dir) / "test"

    # Verify all folders exist
    for d in [train_dir, val_dir, test_dir]:
        if not d.exists():
            raise FileNotFoundError(f"❌ Split folder not found: {d}")

    # Accept uppercase extensions (.JPEG, .PNG, .JPG) as well as lowercase
    VALID_EXTS = {".jpg", ".jpeg", ".png", ".ppm", ".bmp", ".tif", ".tiff", ".webp"}
    def is_valid(path): return Path(path).suffix.lower() in VALID_EXTS

    train_ds = datasets.ImageFolder(train_dir, transform=get_transforms(img_size, True),  is_valid_file=is_valid)
    val_ds   = datasets.ImageFolder(val_dir,   transform=get_transforms(img_size, False), is_valid_file=is_valid)
    test_ds  = datasets.ImageFolder(test_dir,  transform=get_transforms(img_size, False), is_valid_file=is_valid)

    # Verify class names are consistent across splits
    assert train_ds.classes == val_ds.classes == test_ds.classes, \
        f"❌ Class mismatch across splits: {train_ds.classes} vs {val_ds.classes} vs {test_ds.classes}"

    class_names = train_ds.classes

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True,  num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, num_workers=num_workers, pin_memory=True)

    # ── Print split summary ───────────────────────────────────────────────────
    def class_dist(ds):
        from collections import Counter
        counts = Counter(ds.targets)
        return {class_names[k]: v for k, v in sorted(counts.items())}

    print(f"{'─'*54}")
    print(f"  Classes : {class_names}")
    for name, ds in [("Train", train_ds), ("Val", val_ds), ("Test", test_ds)]:
        d    = class_dist(ds)
        dist = "  |  ".join(f"{k}: {v}" for k, v in d.items())
        print(f"  {name:<6}: {len(ds):>5} images   [{dist}]")
    print(f"{'─'*54}\n")

    return train_loader, val_loader, test_loader, class_names, test_ds


train_loader, val_loader, test_loader, CLASS_NAMES, test_subset = build_dataloaders(
    DATA_DIR, IMG_SIZE, BATCH_SIZE, NUM_WORKERS
)
NUM_CLASSES = len(CLASS_NAMES)

# IDX_TRAIN for class weight computation — all training indices
IDX_TRAIN = list(range(len(train_loader.dataset)))


──────────────────────────────────────────────────────
  Classes : ['Normal', 'Pneumonia']
  Train :  5982 images   [Normal: 2991  |  Pneumonia: 2991]
  Val   :   878 images   [Normal: 237  |  Pneumonia: 641]
  Test  :   879 images   [Normal: 238  |  Pneumonia: 641]
──────────────────────────────────────────────────────



### 🖼️ Preview Sample Training Images

In [26]:
mean_np = np.array([0.485, 0.456, 0.406])
std_np  = np.array([0.229, 0.224, 0.225])
images, labels = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
fig.suptitle("Sample training images (after augmentation)", fontsize=12, fontweight="bold")
for i, ax in enumerate(axes.flat):
    if i >= len(images): break
    img = np.clip(images[i].permute(1,2,0).numpy() * std_np + mean_np, 0, 1)
    ax.imshow(img); ax.set_title(CLASS_NAMES[labels[i]], fontsize=8); ax.axis("off")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sample_images.png", dpi=120, bbox_inches="tight")
plt.show()
print("✅ Saved: sample_images.png")


✅ Saved: sample_images.png


## 🧠 Step 6 — Model + Class Weights

ConvNeXt V2 Tiny pretrained on ImageNet-22k. Backbone is frozen for the first `UNFREEZE_EP` epochs — only the classification head trains. Class weights are auto-computed from training labels to correct any residual imbalance.


In [27]:
def build_model(model_name, num_classes, device):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    for name, param in model.named_parameters():
        if "head" not in name:
            param.requires_grad = False
    model = model.to(device)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"  Model      : {model_name}")
    print(f"  Parameters : {total:,} total  |  {trainable:,} trainable (head only)")
    return model


def unfreeze_backbone(model, lr):
    for p in model.parameters():
        p.requires_grad = True
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    new_opt   = optim.AdamW(model.parameters(), lr=lr * 0.1, weight_decay=1e-4)
    print(f"\n✅ Backbone unfrozen — {trainable:,} params  |  fine-tune LR: {lr*0.1:.2e}")
    return new_opt


model = build_model(MODEL_NAME, NUM_CLASSES, DEVICE)

# ── Class-weighted loss ────────────────────────────────────────────────────────
_full_ds_tmp  = datasets.ImageFolder(DATA_DIR)
train_labels  = [_full_ds_tmp.samples[i][1] for i in IDX_TRAIN]
class_weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels)
weights_tensor = torch.FloatTensor(class_weights).to(DEVICE)

w_str = "  |  ".join(f"{CLASS_NAMES[i]}: {w:.4f}x" for i, w in enumerate(class_weights))
print(f"\n  Class weights : [{w_str}]")
print("  (weights near 1.0 = dataset already balanced)")

criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
optimizer  = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                         lr=LR, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
print("\n✅ Model, weighted loss, optimizer and scheduler ready")


  Model      : convnextv2_base.fcmae_ft_in1k
  Parameters : 87,694,850 total  |  4,098 trainable (head only)

  Class weights : [Normal: 3.4027x  |  Pneumonia: 0.5861x]
  (weights near 1.0 = dataset already balanced)

✅ Model, weighted loss, optimizer and scheduler ready


## 💾 Step 7 — Checkpoints

Auto-resumes from `latest.pt` if the session was interrupted.

In [28]:
CKPT_DIR = Path(OUTPUT_DIR) / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def save_checkpoint(state, path):
    torch.save(state, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None):
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    if optimizer and "optimizer_state" in ckpt:
        optimizer.load_state_dict(ckpt["optimizer_state"])
    if scheduler and "scheduler_state" in ckpt:
        scheduler.load_state_dict(ckpt["scheduler_state"])
    return ckpt.get("epoch", 0), ckpt.get("best_val_auc", 0.0)


latest_ckpt  = CKPT_DIR / "latest.pt"
start_epoch  = 0
best_val_auc = 0.0

if latest_ckpt.exists():
    # ── Verify checkpoint matches current model before loading ────────────────
    ckpt = torch.load(latest_ckpt, map_location="cpu", weights_only=False)
    saved_model = ckpt.get("model_name", "unknown")
    current_model = MODEL_NAME

    if saved_model != current_model:
        print(f"⚠️  Checkpoint mismatch!")
        print(f"   Saved model    : {saved_model}")
        print(f"   Current model  : {current_model}")
        print(f"   Deleting stale checkpoint and starting fresh...")
        latest_ckpt.unlink()                          # delete latest.pt
        best_ckpt = CKPT_DIR / "best.pt"
        if best_ckpt.exists():
            best_ckpt.unlink()                        # delete best.pt too
        print("🆕 Stale checkpoints removed — starting fresh")
    else:
        print(f"🔄 Resuming from checkpoint ({saved_model})...")
        start_epoch, best_val_auc = load_checkpoint(latest_ckpt, model, optimizer, scheduler)
        model.to(DEVICE)
        print(f"   Epoch {start_epoch}  |  best AUC so far: {best_val_auc:.4f}")
else:
    print("🆕 No checkpoint found — starting fresh")

🆕 No checkpoint found — starting fresh


## 🚀 Step 8 — Training Loop

Backbone unfreezes at epoch `UNFREEZE_EP` with LR dropped 10×. If Colab disconnects, re-run Steps 7 → 8 — auto-resumes from `latest.pt`.


In [ ]:
def run_epoch(model, loader, criterion, optimizer, device, is_train):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_probs, all_labels = [], []
    ctx   = torch.enable_grad() if is_train else torch.no_grad()
    label = "Train" if is_train else "Val  "
    with ctx:
        for images, labels in tqdm(loader, desc=f"  {label}", leave=False):
            images, labels = images.to(device), labels.to(device)
            if is_train: optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            if is_train: loss.backward(); optimizer.step()
            total_loss += loss.item() * images.size(0)
            probs   = torch.softmax(outputs, dim=1)
            preds   = probs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += images.size(0)
            all_probs.extend(probs.detach().cpu().numpy())
            all_labels.extend(labels.detach().cpu().numpy())
    all_probs  = np.array(all_probs)
    all_labels = np.array(all_labels)
    auc = roc_auc_score(all_labels, all_probs[:,1]) if all_probs.shape[1]==2           else roc_auc_score(all_labels, all_probs, multi_class="ovr")
    return total_loss / total, correct / total, auc


history = {k: [] for k in ["train_loss","val_loss","train_acc","val_acc","train_auc","val_auc"]}

print(f"{'═'*62}")
print(f"  Training  |  epochs: {EPOCHS}  |  unfreeze backbone at: epoch {UNFREEZE_EP}")
print(f"{'═'*62}\n")

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()

    if epoch == UNFREEZE_EP:
        optimizer = unfreeze_backbone(model, LR)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - epoch, eta_min=1e-7)

    tr_loss, tr_acc, tr_auc = run_epoch(model, train_loader, criterion, optimizer, DEVICE, True)
    vl_loss, vl_acc, vl_auc = run_epoch(model, val_loader,   criterion, None,      DEVICE, False)
    scheduler.step()

    for k, v in zip(history, [tr_loss, vl_loss, tr_acc, vl_acc, tr_auc, vl_auc]):
        history[k].append(v)

    star = "⭐" if vl_auc > best_val_auc else "  "
    print(f"{star} [{epoch+1:03d}/{EPOCHS}]  "
          f"Loss {tr_loss:.4f}/{vl_loss:.4f}  "
          f"Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}%  "
          f"AUC {tr_auc:.4f}/{vl_auc:.4f}  "
          f"LR {optimizer.param_groups[0]['lr']:.1e}  "
          f"{time.time()-t0:.0f}s")

    ckpt = dict(epoch=epoch+1,
                model_state=model.state_dict(),
                optimizer_state=optimizer.state_dict(),
                scheduler_state=scheduler.state_dict(),
                best_val_auc=best_val_auc,
                class_names=CLASS_NAMES,
                model_name=MODEL_NAME)
    save_checkpoint(ckpt, CKPT_DIR / "latest.pt")

    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        ckpt["best_val_auc"] = best_val_auc
        save_checkpoint(ckpt, CKPT_DIR / "best.pt")

print(f"\n✅ Training done  |  Best val AUC: {best_val_auc:.4f}")


══════════════════════════════════════════════════════════════
  Training  |  epochs: 30  |  unfreeze backbone at: epoch 5
══════════════════════════════════════════════════════════════



  Train:   0%|          | 0/187 [00:00<?, ?it/s]

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

⭐ [001/30]  Loss 0.3646/0.9207  Acc 52.6%/44.0%  AUC 0.8335/0.9040  LR 3.0e-04  1387s


  Train:   0%|          | 0/187 [00:00<?, ?it/s]

...

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

   [029/30]  Loss 0.1457/0.5447  Acc 99.9%/97.5%  AUC 1.0000/0.9937  LR 2.2e-07  362s


  Train:   0%|          | 0/187 [00:00<?, ?it/s]

  Val  :   0%|          | 0/28 [00:00<?, ?it/s]

   [030/30]  Loss 0.1466/0.5444  Acc 99.9%/97.3%  AUC 1.0000/0.9939  LR 1.0e-07  364s

✅ Training done  |  Best val AUC: 0.9982


## 📈 Step 9 — Training Curves

In [ ]:
ep = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"Training history — {MODEL_NAME}", fontsize=13, fontweight="bold")
for ax, (tk, vk), title, color in zip(
    axes,
    [("train_loss","val_loss"),("train_acc","val_acc"),("train_auc","val_auc")],
    ["Loss","Accuracy","AUC"],
    ["#e74c3c","#2ecc71","#3498db"]
):
    ax.plot(ep, history[tk], label="Train", color=color, linewidth=2)
    ax.plot(ep, history[vk], label="Val",   color=color, linewidth=2, linestyle="--")
    if title == "Accuracy":
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f"{y*100:.0f}%"))
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("📈 Saved: training_curves.png")

# ── CSV export: training history ─────────────────────────────────────────────
import pandas as pd
history_df = pd.DataFrame({
    "epoch":      list(ep),
    "train_loss": history["train_loss"],
    "val_loss":   history["val_loss"],
    "train_acc":  history["train_acc"],
    "val_acc":    history["val_acc"],
    "train_auc":  history["train_auc"],
    "val_auc":    history["val_auc"],
})
history_csv = f"{OUTPUT_DIR}/training_history.csv"
history_df.to_csv(history_csv, index=False)
print(f"📄 Saved CSV: training_history.csv")
print(history_df.to_string(index=False))


📈 Saved: training_curves.png
📄 Saved CSV: training_history.csv
 epoch  train_loss  val_loss  train_acc  val_acc  train_auc  val_auc
     1    0.364582  0.920675   0.525577 0.439636   0.833509 0.903974
     2    0.273577  0.906116   0.702273 0.547836   0.945620 0.919410
     3    0.254200  0.886722   0.755433 0.576310   0.956158 0.930870
     4    0.244115  0.929897   0.777165 0.543280   0.964177 0.937913
     5    0.238584  0.918248   0.784019 0.544419   0.968774 0.943752
     6    0.195052  0.889834   0.908559 0.743736   0.988584 0.994293
     7    0.176002  0.687676   0.944835 0.839408   0.994896 0.996590
     8    0.166644  0.574707   0.966065 0.938497   0.997040 0.997097
     9    0.169877  0.685223   0.964226 0.875854   0.996063 0.995741
    10    0.161923  0.607703   0.973755 0.907745   0.997714 0.998183
    11    0.162250  0.652506   0.976429 0.886105   0.997808 0.993681
    12    0.160027  0.578148   0.976095 0.940774   0.998692 0.995695
    13    0.157121  0.558865   0.981110 

## 🧪 Step 10 — Test Set Evaluation

In [ ]:
print("Loading best checkpoint...")
load_checkpoint(CKPT_DIR / "best.pt", model)
model.to(DEVICE).eval()

all_preds, all_probs_list, all_labels_list = [], [], []
total_loss, correct, total = 0.0, 0, 0

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="  Test"):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss    = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        probs   = torch.softmax(outputs, dim=1)
        preds   = probs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total   += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_probs_list.extend(probs.cpu().numpy())
        all_labels_list.extend(labels.cpu().numpy())

all_preds  = np.array(all_preds)
all_probs  = np.array(all_probs_list)
all_labels = np.array(all_labels_list)

test_acc = correct / total
test_f1  = f1_score(all_labels, all_preds, average="weighted")
test_auc = roc_auc_score(all_labels, all_probs[:,1]) if NUM_CLASSES==2            else roc_auc_score(all_labels, all_probs, multi_class="ovr")

print(f"\n{'═'*52}")
print(f"  TEST RESULTS")
print(f"{'═'*52}")
print(f"  Accuracy  : {test_acc*100:.2f}%")
print(f"  AUC       : {test_auc:.4f}")
print(f"  F1        : {test_f1:.4f}")
print(f"  Loss      : {total_loss/total:.4f}")
print(f"{'═'*52}")
print(f"\n{classification_report(all_labels, all_preds, target_names=CLASS_NAMES)}")

# ── CSV export: overall test metrics ──────────────────────────────────────────
import pandas as pd
metrics_df = pd.DataFrame([{
    "model":        MODEL_NAME,
    "test_accuracy": round(test_acc, 6),
    "test_auc":      round(test_auc, 6),
    "test_f1":       round(test_f1, 6),
    "test_loss":     round(total_loss / total, 6),
    "n_samples":     total,
}])
metrics_csv = f"{OUTPUT_DIR}/test_metrics.csv"
metrics_df.to_csv(metrics_csv, index=False)
print(f"📄 Saved CSV: test_metrics.csv")

# ── CSV export: per-class classification report ────────────────────────────────
from sklearn.metrics import classification_report
report_dict = classification_report(
    all_labels, all_preds, target_names=CLASS_NAMES, output_dict=True
)
report_df = pd.DataFrame(report_dict).transpose().reset_index()
report_df.rename(columns={"index": "class"}, inplace=True)
report_csv = f"{OUTPUT_DIR}/classification_report.csv"
report_df.to_csv(report_csv, index=False)
print(f"📄 Saved CSV: classification_report.csv")
print(report_df.to_string(index=False))


Loading best checkpoint...


  Test:   0%|          | 0/28 [00:00<?, ?it/s]


════════════════════════════════════════════════════
  TEST RESULTS
════════════════════════════════════════════════════
  Accuracy  : 91.13%
  AUC       : 0.9945
  F1        : 0.9146
  Loss      : 0.6493
════════════════════════════════════════════════════

              precision    recall  f1-score   support

      Normal       0.75      1.00      0.86       238
   Pneumonia       1.00      0.88      0.94       641

    accuracy                           0.91       879
   macro avg       0.88      0.94      0.90       879
weighted avg       0.93      0.91      0.91       879

📄 Saved CSV: test_metrics.csv
📄 Saved CSV: classification_report.csv
       class  precision   recall  f1-score    support
      Normal   0.754777 0.995798  0.858696 238.000000
   Pneumonia   0.998230 0.879875  0.935323 641.000000
    accuracy   0.911263 0.911263  0.911263   0.911263
   macro avg   0.876504 0.937837  0.897010 879.000000
weighted avg   0.932312 0.911263  0.914575 879.000000


### 🔲 Confusion Matrix

In [ ]:
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, data, fmt, title in zip(
    axes, [cm, cm_norm], ["d", ".2f"],
    ["Counts", "Normalized"]
):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.set_title(f"Confusion matrix — {title}", fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("📊 Saved: confusion_matrix.png")

# ── CSV export: confusion matrix (counts) ────────────────────────────────────
import pandas as pd
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
cm_df.index.name = "true \ predicted"
cm_csv = f"{OUTPUT_DIR}/confusion_matrix.csv"
cm_df.to_csv(cm_csv)
print(f"📄 Saved CSV: confusion_matrix.csv")
print(cm_df.to_string())

# ── CSV export: confusion matrix (normalized) ─────────────────────────────────
cm_norm_df = pd.DataFrame(cm_norm.round(4), index=CLASS_NAMES, columns=CLASS_NAMES)
cm_norm_df.index.name = "true \ predicted"
cm_norm_csv = f"{OUTPUT_DIR}/confusion_matrix_normalized.csv"
cm_norm_df.to_csv(cm_norm_csv)
print(f"📄 Saved CSV: confusion_matrix_normalized.csv")


📊 Saved: confusion_matrix.png
📄 Saved CSV: confusion_matrix.csv
                  Normal  Pneumonia
true \ predicted                   
Normal               237          1
Pneumonia             77        564
📄 Saved CSV: confusion_matrix_normalized.csv


### 📉 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
if NUM_CLASSES == 2:
    fpr, tpr, _ = roc_curve(all_labels, all_probs[:,1])
    ax.plot(fpr, tpr, linewidth=2.5, color="#2ecc71",
            label=f"AUC = {roc_auc_score(all_labels, all_probs[:,1]):.4f}")
else:
    for i, (name, color) in enumerate(zip(CLASS_NAMES, plt.cm.tab10.colors)):
        fpr, tpr, _ = roc_curve((all_labels==i).astype(int), all_probs[:,i])
        auc_i = roc_auc_score((all_labels==i).astype(int), all_probs[:,i])
        ax.plot(fpr, tpr, linewidth=2, color=color, label=f"{name}  AUC={auc_i:.3f}")
ax.plot([0,1],[0,1],"k--",linewidth=1,label="Random")
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.set_title("ROC curve", fontweight="bold")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("📉 Saved: roc_curve.png")

# ── CSV export: ROC curve data points ─────────────────────────────────────────
import pandas as pd
roc_rows = []
if NUM_CLASSES == 2:
    fpr_vals, tpr_vals, thresholds = roc_curve(all_labels, all_probs[:,1])
    for f, t, th in zip(fpr_vals, tpr_vals, thresholds):
        roc_rows.append({"class": CLASS_NAMES[1], "fpr": round(f, 6),
                         "tpr": round(t, 6), "threshold": round(float(th), 6)})
else:
    for i, name in enumerate(CLASS_NAMES):
        fpr_vals, tpr_vals, thresholds = roc_curve((all_labels==i).astype(int), all_probs[:,i])
        for f, t, th in zip(fpr_vals, tpr_vals, thresholds):
            roc_rows.append({"class": name, "fpr": round(f, 6),
                             "tpr": round(t, 6), "threshold": round(float(th), 6)})
roc_df = pd.DataFrame(roc_rows)
roc_csv = f"{OUTPUT_DIR}/roc_curve.csv"
roc_df.to_csv(roc_csv, index=False)
print(f"📄 Saved CSV: roc_curve.csv  ({len(roc_df)} data points)")
print(roc_df.head(10).to_string(index=False))


📉 Saved: roc_curve.png
📄 Saved CSV: roc_curve.csv  (40 data points)
    class      fpr      tpr  threshold
Pneumonia 0.000000 0.000000        inf
Pneumonia 0.000000 0.001560   0.823442
Pneumonia 0.000000 0.485179   0.711819
Pneumonia 0.000000 0.488300   0.711763
Pneumonia 0.000000 0.514821   0.709781
Pneumonia 0.000000 0.517941   0.709708
Pneumonia 0.000000 0.858034   0.552599
Pneumonia 0.004202 0.858034   0.547218
Pneumonia 0.004202 0.907956   0.322507
Pneumonia 0.008403 0.907956   0.313292


## 🔥 Step 11 — Grad-CAM Visualizations

Balanced sampling — exactly `GRADCAM_SAMPLES // 2` images per class. Green title = correct prediction, Red = wrong.


In [ ]:
if not GRADCAM_AVAILABLE:
    print("⚠️  Grad-CAM not available — re-run Step 1 and restart kernel")
else:
    # ── Find last Conv2d layer dynamically ────────────────────────────────────
    target_layers  = []
    last_conv_name = ""
    for name, module in model.named_modules():
        if isinstance(module, nn.Conv2d):
            target_layers  = [module]
            last_conv_name = name
    if not target_layers:
        print("❌ No Conv2d layer found — cannot run Grad-CAM")
    else:
        print(f"✅ Grad-CAM target layer: {last_conv_name}")

        # ── Balanced class sampling ───────────────────────────────────────────
        class_indices = defaultdict(list)
        for idx in range(len(test_subset)):
            _, lbl = test_subset[idx]
            class_indices[lbl].append(idx)

        n_per_class = GRADCAM_SAMPLES // NUM_CLASSES
        indices = []
        for cls in sorted(class_indices.keys()):
            chosen = random.sample(class_indices[cls], min(n_per_class, len(class_indices[cls])))
            indices.extend(chosen)
            print(f"  Sampling {len(chosen)} images from: {CLASS_NAMES[cls]}")
        random.shuffle(indices)
        n = len(indices)

        # ── Build CAM and plot ────────────────────────────────────────────────
        cam = GradCAM(model=model, target_layers=target_layers)
        fig, axes = plt.subplots(2, n, figsize=(n * 3, 6))
        fig.suptitle("Grad-CAM  |  Row 1: original   Row 2: activation heatmap",
                     fontsize=12, fontweight="bold")

        for col, idx in enumerate(indices):
            img_t, label = test_subset[idx]
            inp = img_t.unsqueeze(0).to(DEVICE)
            inp.requires_grad_(True)       # Grad-CAM needs gradient tracking

            model.eval()                   # eval mode — no no_grad wrapper
            pred = model(inp).argmax(1).item()

            gcam = cam(input_tensor=inp, targets=[ClassifierOutputTarget(pred)])[0]

            # Denormalize for display
            mean_np = np.array([0.485, 0.456, 0.406])
            std_np  = np.array([0.229, 0.224, 0.225])
            img_np  = img_t.permute(1,2,0).numpy()
            img_np  = np.clip(img_np * std_np + mean_np, 0, 1).astype(np.float32)
            cam_img = show_cam_on_image(img_np, gcam, use_rgb=True)

            axes[0, col].imshow(img_np)
            axes[0, col].set_title(f"True:\n{CLASS_NAMES[label]}", fontsize=8)
            axes[0, col].axis("off")

            axes[1, col].imshow(cam_img)
            axes[1, col].set_title(f"Pred:\n{CLASS_NAMES[pred]}", fontsize=8,
                                   color="green" if pred==label else "red")
            axes[1, col].axis("off")


        # ── CSV export: Grad-CAM per-image predictions ───────────────────────────
        import pandas as pd
        gradcam_records = []
        for col, idx in enumerate(indices):
            img_t2, label2 = test_subset[idx]
            inp2 = img_t2.unsqueeze(0).to(DEVICE)
            with torch.no_grad():
                pred2 = model(inp2).argmax(1).item()
                prob2 = torch.softmax(model(inp2), dim=1)[0].cpu().numpy()
            gradcam_records.append({
                "sample_index":   idx,
                "true_label":     CLASS_NAMES[label2],
                "pred_label":     CLASS_NAMES[pred2],
                "correct":        label2 == pred2,
                **{f"prob_{CLASS_NAMES[c]}": round(float(prob2[c]), 6) for c in range(NUM_CLASSES)},
            })
        gradcam_df = pd.DataFrame(gradcam_records)
        gradcam_csv = f"{OUTPUT_DIR}/gradcam_predictions.csv"
        gradcam_df.to_csv(gradcam_csv, index=False)
        print(f"📄 Saved CSV: gradcam_predictions.csv  ({len(gradcam_df)} samples)")
        print(gradcam_df.to_string(index=False))
        plt.tight_layout()
        plt.savefig(f"{OUTPUT_DIR}/gradcam.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"🔥 Saved: gradcam.png  ({n_per_class} per class)")


## ✅ Step 12 — Summary

In [ ]:
from IPython.display import display, HTML

html = f"""
<div style="font-family:monospace;background:var(--jp-layout-color1,#f8f9fa);
            padding:20px;border-radius:8px;border-left:4px solid #2ecc71">
  <h3 style="margin-top:0;color:#2c3e50">Training complete</h3>
  <table style="width:100%;border-collapse:collapse;font-size:14px">
    <tr><td style="padding:5px;color:#666">Model</td>
        <td><b>{MODEL_NAME}</b></td></tr>
    <tr><td style="padding:5px;color:#666">Test accuracy</td>
        <td style="color:#e74c3c;font-weight:bold">{test_acc*100:.2f}%</td></tr>
    <tr><td style="padding:5px;color:#666">Test AUC</td>
        <td style="color:#3498db;font-weight:bold">{test_auc:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Test F1</td>
        <td style="color:#9b59b6;font-weight:bold">{test_f1:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Best val AUC</td>
        <td>{best_val_auc:.4f}</td></tr>
    <tr><td style="padding:5px;color:#666">Classes</td>
        <td>{CLASS_NAMES}</td></tr>
  </table>
  <br>
  <b>Saved to:</b> {OUTPUT_DIR}/<br>
  <ul style="margin:6px 0;padding-left:20px;font-size:13px">
    <li>checkpoints/best.pt — best model weights</li>
    <li>checkpoints/latest.pt — resume token</li>
    <li>sample_images.png | training_curves.png</li>
    <li>confusion_matrix.png | roc_curve.png | gradcam.png</li>
    <li>training_history.csv | test_metrics.csv | classification_report.csv</li>
    <li>confusion_matrix.csv | confusion_matrix_normalized.csv</li>
    <li>roc_curve.csv | gradcam_predictions.csv | summary.csv</li>
  </ul>
</div>
"""
display(HTML(html))

# ── CSV export: summary of the full run ───────────────────────────────────────
import pandas as pd
summary_df = pd.DataFrame([{
    "model":          MODEL_NAME,
    "classes":        str(CLASS_NAMES),
    "test_accuracy":  round(test_acc, 6),
    "test_auc":       round(test_auc, 6),
    "test_f1":        round(test_f1, 6),
    "best_val_auc":   round(best_val_auc, 6),
    "output_dir":     OUTPUT_DIR,
}])
summary_csv = f"{OUTPUT_DIR}/summary.csv"
summary_df.to_csv(summary_csv, index=False)
print(f"📄 Saved CSV: summary.csv")
print(summary_df.to_string(index=False))
